# Dataset Autocheck: Flagging Suspicious Samples With Self-Supervised Models

Curating large datasets for machine learning by hand is hard.
Broken or noisy data must be kept out as much as possible,
both so training works and so the reported validation statistics can be trusted.

A simple form of anomaly detection is to train a model on the dataset itself and flag the samples it fits worst.
`popsim.data.autocheck` does that for a tensorized dataset (see `popsim.data.dataset_utils.build_tensorized_dataset`)
in two modes:

- **Time-independent** (default): an MLP autoencoder over every `(episode, time)` sample.
  The error is the reconstruction error.
  It catches samples that are off the manifold of the rest of the data, such as a signal that is suddenly negated or scaled.
- **Time-dependent**: a causal transformer over segments of consecutive time steps
  that predicts every signal at step `t + 1` from all signals up to step `t`.
  The error is the one-step-ahead prediction error per time step.
  It catches sequences that are individually plausible but inconsistent in time, such as shuffled or glitching signals.

The score of a sample is the mean over variables of each variable's mean squared normalized error,
so a broken scalar signal counts as much as a broken profile or image.

Scoring is cross-fitted.
The episodes are split into `n_folds` folds, and the model of each fold trains on the other folds.
Each model only scores its held-out fold, so no sample is scored by a model that has seen it.
There is no clean data to validate on either,
so the models train on the robust loss `log(1 + score)`,
which keeps the anomalies in the training and validation episodes from steering the fit or the early stopping.

Model sizes are derived from the data.
The latent size is the number of principal components needed to explain 99% of the variance of the normalized data
(see `popsim.modules.linear_autoencoder.choose_n_latent`),
so the model is neither large enough to fit every quirk nor too small to generalize.

Within each fold the log scores are turned into robust z-scores (median and median absolute deviation),
and samples above a threshold are flagged for a human expert to review.

## A demo dataset with known problems

The demo dataset below has the layout `build_tensorized_dataset` produces
(one episode per zarr chunk, `time` as a NaN-padded data variable).
Every episode has 100 samples of the signals:

| signal | definition |
| --- | --- |
| `signal_x` | the time coordinate |
| `signal_y` | `((3 * episode) + 1) % 2` |
| `signal_a` | `sqrt(y) + x` |
| `signal_b` | `2 a` |
| `signal_c` | `ln(a + 0.1)` |
| `signal_d` | `((3 a) + 1) % 2` |
| `signal_e(rho)` | `a rho^2`, a 20 point profile |
| `signal_f(R, Z)` | `ln(R + a + 0.1) + Z`, an 8 x 8 image |

Episodes come in three groups, numbered in this order:

1. 40 episodes of clean data.
2. 10 episodes of clean data with five aberrations each,
   where one signal is negated, scaled by 4, zeroed or offset for one to three consecutive steps.
   These are time-independent outliers.
3. One episode of clean data whose rows are shuffled in time while the time coordinate stays monotonic.
   Every sample is plausible on its own, only the sequence is wrong.
   This is a time-dependent outlier.

The `aberration_code` variable holds the ground truth per step (see `Aberration`)
and is only used here to check the results.

In [ ]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from popsim.data.autocheck import autocheck
from popsim.data.dataset_utils import build_tensorized_dataset
from popsim.data.dummy.data_generators import (
    ABERRATION_CODE_VAR,
    DEMO_SIGNALS,
    EVENT_ABERRATIONS,
    Aberration,
    generate_autocheck_demo_dataset,
)

N_NORMAL, N_ABERRANT, N_SHUFFLED = 40, 10, 1

work_dir = tempfile.mkdtemp()
zarr_path = os.path.join(work_dir, "autocheck_demo.zarr")

build_tensorized_dataset(
    process_fn=lambda episode_ds: episode_ds,
    identifiers=generate_autocheck_demo_dataset(n_normal=N_NORMAL, n_aberrant=N_ABERRANT, n_shuffled=N_SHUFFLED, n_time=100),
    zarr_path=zarr_path,
    time_dim="time_idx",
    episode_dim="episode",
)
ds = xr.open_zarr(zarr_path).set_coords("time").load()
ds

In [ ]:
EXAMPLE_EPISODES = {"clean": 0, "aberrant": N_NORMAL, "shuffled": N_NORMAL + N_ABERRANT}
SCALAR_SIGNALS = ["signal_a", "signal_b", "signal_c", "signal_d"]


def shade_aberrations(ax, episode):
    # Shade the steps the ground truth marks as aberrant.
    codes = ds[ABERRATION_CODE_VAR].sel(episode=episode).values
    time = ds["time"].sel(episode=episode).values
    for step in np.flatnonzero(np.isin(codes, EVENT_ABERRATIONS)):
        ax.axvspan(time[step] - 0.05, time[step] + 0.05, color="red", alpha=0.2, lw=0)


fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (label, episode) in zip(axes, EXAMPLE_EPISODES.items()):
    for signal in SCALAR_SIGNALS:
        ax.plot(ds["time"].sel(episode=episode), ds[signal].sel(episode=episode), label=signal)
    shade_aberrations(ax, episode)
    ax.set_title(f"{label} episode {episode}")
    ax.set_xlabel("time")
axes[0].legend()
plt.show()

## Time-independent mode

`autocheck` opens the dataset and splits the episodes into folds.
For every fold it splits the other folds into training and validation episodes,
trains the model with early stopping, restores the best checkpoint and scores the held-out fold.
It then writes `autocheck_scores.nc`, `flagged.csv` and `episode_summary.csv` to the output directory.
The logged tables are the same ones the command line prints.

In [ ]:
result_indep = autocheck(
    zarr_path,
    os.path.join(work_dir, "autocheck_time_indep"),
    mode="time_indep",
    variables=list(DEMO_SIGNALS),
    episode_coord="episode",
    max_epochs=150,
    patience=20,
    batch_size=128,
)

`result.scores` is a Dataset with one row per scored sample:
the error of every variable, the score, its robust z-score, the flag, the variable with the largest error and the fold.
Without the per-variable errors it becomes a table.
The top flagged rows point straight at the injected aberrations.

In [ ]:
DISPLAY_COLUMNS = ["episode", "time", "score", "z_score", "worst_variable", "worst_variable_error", "fold"]

scores_indep = result_indep.scores.drop_dims("variable").to_dataframe()
scores_indep[scores_indep["flagged"]].sort_values("z_score", ascending=False)[DISPLAY_COLUMNS].head(15)

### Comparing with the ground truth

Join the scores with `aberration_code`.
Recall counts the injected aberration steps that were flagged,
and the false positive rate counts the clean steps that were flagged.
The shuffled episode is not expected to be caught in this mode,
since every one of its samples is a valid point of the clean manifold.

In [ ]:
truth = ds[ABERRATION_CODE_VAR].to_dataframe().reset_index()[["episode", "time", ABERRATION_CODE_VAR]]


def with_ground_truth(result):
    scores = result.scores.drop_dims("variable").to_dataframe()
    return scores.merge(truth, on=["episode", "time"], how="left")


def report_detection(scores):
    codes = scores[ABERRATION_CODE_VAR]
    mask_aberrant = codes.isin(EVENT_ABERRATIONS)
    mask_clean = codes == Aberration.NONE
    mask_shuffled = codes == Aberration.SHUFFLED
    print(f"Recall on aberrant steps:          {scores.loc[mask_aberrant, 'flagged'].mean():.2f}  ({mask_aberrant.sum()} steps)")
    print(f"False positive rate on clean steps: {scores.loc[mask_clean, 'flagged'].mean():.3f}  ({mask_clean.sum()} steps)")
    print(f"Flagged fraction of shuffled steps: {scores.loc[mask_shuffled, 'flagged'].mean():.2f}  ({mask_shuffled.sum()} steps)")


scores_indep = with_ground_truth(result_indep)
report_detection(scores_indep)

In [ ]:
def plot_zscores(scores, title):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, (label, episode) in zip(axes, EXAMPLE_EPISODES.items()):
        rows = scores[scores["episode"] == episode].sort_values("time")
        ax.plot(rows["time"], rows["z_score"], ".-")
        ax.axhline(3.5, color="black", ls="--", lw=1, label="z threshold")
        shade_aberrations(ax, episode)
        ax.set_title(f"{label} episode {episode}")
        ax.set_xlabel("time")
    axes[0].set_ylabel("robust z-score")
    axes[0].legend()
    fig.suptitle(title)
    plt.show()


plot_zscores(scores_indep, "Time-independent mode")

Most aberrations stand out by several robust standard deviations.
The misses are the subtle ones,
for example zeroing a signal at a time where it is already close to zero,
or scaling `signal_d` where the modulo wraps the result back onto plausible values.
The sawtooth discontinuities of `signal_d` can also produce false positives,
which is a fair illustration of the limits of the method: the model can only flag what is rare in the data it was given.

## Time-dependent mode

The time-dependent loader cuts every episode into segments of 40 steps that overlap by 20 (`make_time_dep_dataloader`),
and the causal transformer predicts each step from the steps before it.
Every step gets its own score, so a transient is localized in time.
Where overlapping segments score the same step twice, the segment giving the step the most context wins.
The overlap must be at least one step, so that the first step of a segment is scored by the segment before it.

Segments are few per episode (about five here), so training uses a small batch size and more epochs than
the time-independent run.

In [ ]:
SEGMENT_LENGTH = 40

result_dep = autocheck(
    zarr_path,
    os.path.join(work_dir, "autocheck_time_dep"),
    mode="time_dep",
    variables=list(DEMO_SIGNALS),
    episode_coord="episode",
    segment_length=SEGMENT_LENGTH,
    segment_overlap=20,
    max_epochs=300,
    patience=30,
    batch_size=16,
)

In [ ]:
scores_dep = with_ground_truth(result_dep)
report_detection(scores_dep)
result_dep.episode_summary.head(10)

In [ ]:
plot_zscores(scores_dep, "Time-dependent mode")

The shuffled episode, invisible to the autoencoder, now stands out: none of its steps follow from the previous ones.
The aberrations are still caught,
and the step right after an aberration often scores high too, since the model then predicts from a corrupted context.

### Looking at a prediction

The trained scorer of every fold is on the result, and `episode_summary` holds the fold of each episode.
Below, the one-step-ahead prediction of `signal_a` is compared with the data
for the first segment of a clean episode and of the shuffled episode,
each by the model that never saw it in training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, label in zip(axes, ["clean", "shuffled"]):
    episode = EXAMPLE_EPISODES[label]
    fold = result_dep.episode_summary.loc[episode, "fold"]
    predictor = result_dep.scorers[fold].model
    segment = ds[list(DEMO_SIGNALS)].sel(episode=episode).isel(time_idx=slice(0, SEGMENT_LENGTH))
    time = segment["time"].values
    predicted = predictor(segment, time)
    ax.plot(time, segment["signal_a"].values, "o-", label="data")
    ax.plot(time, np.asarray(predicted["signal_a"].data), "x--", label="one-step prediction")
    ax.set_title(f"{label} episode {episode}, first segment")
    ax.set_xlabel("time")
axes[0].set_ylabel("signal_a")
axes[0].legend()
plt.show()

## Running on a real dataset

Both modes are available from the command line, `python -m popsim.data.autocheck` or the `popsim-autocheck` script:

```
popsim-autocheck path/to/dataset.zarr out_dir
popsim-autocheck path/to/dataset.zarr out_dir --mode time_dep --segment_length 40 --segment_overlap 20
popsim-autocheck path/to/dataset.zarr out_dir --variables "LY.Ip,LY.bp,LY.Ia" --z_threshold 4 --n_folds 10
```

By default every numeric variable with both the episode and time dimensions is modeled.
Things to keep in mind:

- Training takes `n_folds` times as long as a single model.
  More folds train every model on more episodes, fewer folds are cheaper.
- Within a fold the z-scores assume anomalies are rare.
  An anomaly that covers a large share of a fold's samples raises that fold's median and scale and can hide itself.
- The dataset is loaded into memory.
  Time-dependent mode needs about `segment_length / (segment_length - segment_overlap)` times the data size on top of that.
- Samples with a NaN in any modeled variable are not scored.
  The `n_unscored` column of `episode_summary.csv` counts the non-NaN times without a score per episode,
  and episodes without any scoreable sample are listed in the log.
- The first steps of an episode have little context in time-dependent mode, so their scores are less reliable.
  The very first step is never scored, and forward-filled tails of partial segments are masked out entirely.
- `z_threshold` trades recall for review effort.
  `flagged.csv` is sorted by z-score, so reviewing from the top is a reasonable way to spend a limited budget.